# EarlySign: Group Sequential Design (GSD) Demonstration

This notebook demonstrates the use of the `EarlySign` library for **Group Sequential Design (GSD)** in A/B testing.

We will explore two types of designs:
1.  **Non-Binding Futility**: Designed to allow stopping for futility, but the **Type I error control (and ASN calculation)** assumes we ignore these stops. This results in a "High ASN" curve at the Null.
2.  **Binding Futility**: The standard efficient design. We commit to stopping, allowing for aggressive sample size reduction under the Null (the "Rising ASN" curve).


## 1. Design A: Non-Binding Futility (Intuitive)

First, we design a protocol where futility boundaries are **Non-Binding**. This means that even if we cross the futility boundary (stopping for "failure"), it does not statistically penalize our ability to declare success later (Type I error is controlled assuming we *continue*).

**Scenario**:
*   **Control CTR**: 2.0%
*   **Treatment CTR**: 2.1% (Relative Lift: +5%)
*   **Futility**: Non-Binding (`binding=False`)

This configuration is often preferred when you want the option to "override" a futility signal without invalidating the test's statistical guarantees.


In [ ]:
import ibis
import matplotlib.pyplot as plt
from earlysign.core.ledger import Ledger
from earlysign.v1.templates.binomial_ab import BinomialABTemplate, BinomialABTaskSpec
import earlysign.schema.ES3.GST as GST


# 1. Define Task (Non-Binding)
task_nb = BinomialABTaskSpec(
    arms=["control", "treatment"],
    efficacy=GST.EfficacyRequirement(alpha=0.05),
    futility=GST.FutilityRequirement(power=0.8, binding=False),  # <--- NON-BINDING
    hypotheses=GST.HypothesisSpec(
        h_null_description="Difference <= 0",
        h_alt_description="Difference > 0.001",
        test_logic=GST.SuperiorityHypothesis(superiority_margin=0.0),
        target_effect=GST.BinaryEffectSize(
            proportions={"control": 0.02, "treatment": 0.021}
        ),
    ),
)

# Design Protocol (3 Looks)
protocol_nb = BinomialABTemplate.design(
    task=task_nb,
    looks=3,
    spending_function="obrien_fleming",
    designer_params={"model": "canonical_joint", "model_params": {"rng_seed": 42}},
)

max_n_nb = int(protocol_nb.method.stopping_policy.timer.max_sample_size)
print(f"Non-Binding Max N: {max_n_nb:,}")

### Visualizing Non-Binding Efficiency
The plot below shows the **Conservative ASN Curve** (ignoring futility stops).

*   **Behavior**: Because the design is "Non-Binding", the Type I error must be controlled assuming we *never* stop for futility. Thus, the "planned" sample size remains high (near Max N) even under the Null hypothesis.
*   **Comparison**: Contrast this with the Binding design below, which gets "credit" for stopping early.


In [ ]:
# --- Visualization: Non-Binding Design ---
from earlysign.v1.methods.group_sequential.plan.operating_characteristics import (
    BinomialABOperatingCharacteristicsEvaluator,
)
from earlysign.v1.methods.group_sequential.plan.plots import (
    plot_operating_characteristics,
    overlay_fixed_design_reference,
)
from earlysign.v1.methods.group_sequential.plan.fixed_sample_design import (
    calculate_fixed_sample_size,
)
from scipy.stats import norm
import matplotlib.pyplot as plt

# 1. Simulate Operating Characteristics
# We evaluate over +/- 50% relative lift range
sim_nb = BinomialABOperatingCharacteristicsEvaluator(protocol_nb, method="simulation")
# For Non-Binding, we visualize the "Conservative" ASN (ignoring futility stops)
# Note: Evaluator respects protocol binding.
# Protocol A is Non-Binding, so simulation will reflect that naturally?
# Actually Evaluator -> AsymptoticSimulator -> CanonicalJointModel
# CanonicalJointModel respects config.efficacy_binding/futility_binding from protocol.
# So we don't need 'ignore_futility=True' param if protocol is set up correctly.
# Evaluator doesn't support 'ignore_futility' arg in evaluate_lift_curve currently.
res_nb = sim_nb.evaluate_lift_curve(
    range_min=-0.5, range_max=0.5, metric_type="relative_lift_pct"
)

# 2. Calculate Fixed Design Reference (Total N)
# Note: fixed_sample_design uses pooled variance approximation formula (Total N)
n_fixed = calculate_fixed_sample_size(
    p_control=0.02, p_treatment=0.021, alpha=0.05, power=0.8, sided=1
)

# 3. Plot
fig, ax = plt.subplots(figsize=(10, 6))
plot_operating_characteristics(
    res_nb,
    ax=ax,
    title="Design A: Non-Binding Futility (Evaluated Ignoring Futility Stops)",
)
overlay_fixed_design_reference(
    ax, res_nb.target_x_value, n_fixed, label="Fixed Design N (Total)"
)
plt.show()

## 2. Design B: Binding Futility (Efficient)

Now, we design with **Binding Futility**. This is the standard "efficient" design.
*   **Futility**: Binding (`binding=True`)

Here, we commit to stopping if the futility boundary is crossed. This commitment forces "Type I Error" spending to account for these stops, allowing us to lower the efficacy boundaries slightly or otherwise optimize the max sample size.


In [ ]:
# Define Task (Binding)
# Define Task (Binding)
task_binding = BinomialABTaskSpec(
    arms=["control", "treatment"],
    efficacy=GST.EfficacyRequirement(alpha=0.05),
    futility=GST.FutilityRequirement(power=0.8, binding=True),
    hypotheses=GST.HypothesisSpec(
        h_null_description="Difference <= 0",
        h_alt_description="Difference > 0.001",
        test_logic=GST.SuperiorityHypothesis(superiority_margin=0.0),
        target_effect=GST.BinaryEffectSize(
            proportions={"control": 0.02, "treatment": 0.021}
        ),
    ),
)

# Design Protocol (3 Looks)
# Using Power Family (standard for GSD)
protocol_binding = BinomialABTemplate.design(
    task=task_binding,
    looks=3,
    spending_function="power_family",
    designer_params={"model": "canonical_joint", "model_params": {"rng_seed": 42}},
)

max_n_binding = int(protocol_binding.method.stopping_policy.timer.max_sample_size)
print(f"Binding Max N:    {max_n_binding:,}")
print(f"Savings vs NB:    {max_n_nb - max_n_binding:,} samples")

### Visualizing Binding Efficiency (Standard)

The plot below shows the **Operating Characteristics** of the Binding design.

#### Note on "Max N Inflation"
You may observe that the **Max Sample Size (GSD)** is slightly larger than the **Fixed Design Sample Size** (Reference Line). This "inflation" is the statistical cost of having the option to stop early.
*   **O'Brien-Fleming (Default)**: We use the standard O'Brien-Fleming spending function, which is designed to minimize this inflation (keeping Max N typically within 1-5% of Fixed N) by spending very little alpha at early looks.

#### Note on Naming Confusion (Per-Arm vs Total N)
*   **Fixed Design N**: Standard sample size formulas (and online calculators) often output **N per arm**.
*   **GSD Max N**: The `EarlySign` library reports **Total Sample Size** (Control + Treatment).
*   *Correction*: The gray reference line below now correctly plots the **Total Fixed Sample Size**, showing that the actual inflation is small.


In [ ]:
# --- Visualization: Binding Design ---
# 1. Simulate
sim_b = BinomialABOperatingCharacteristicsEvaluator(
    protocol_binding, method="simulation"
)
res_b = sim_b.evaluate_lift_curve(
    range_min=-0.5, range_max=0.5, metric_type="relative_lift_pct"
)

# 2. Plot
fig, ax = plt.subplots(figsize=(10, 6))
plot_operating_characteristics(
    res_b, ax=ax, title="Design B: Binding Futility (Efficient)"
)
# Reuse n_fixed from above as it represents the same scientific goal
overlay_fixed_design_reference(
    ax, res_b.target_x_value, n_fixed, label="Fixed Design N (Total)"
)
plt.show()

## 3. Simulation 1: Realistic Lift (Matches Design)

We will use **Design B (Binding)** for our simulations, as it is the more efficient choice.

Scenario:
*   **Control**: 2.0%
*   **Treatment**: 2.1% (+5% lift)


In [ ]:
from earlysign.v1.tests.util import BinomialStream
from earlysign.schema.ES3.GST.Log import DecisionStatus

# Setup In-memory DB
conn = ibis.connect("duckdb://:memory:")
ledger = Ledger(conn, "events_sim1")
ledger.ensure()
ledger = ledger.bind(experiment_id="sim_real_001")

# Initialize Template with Binding Protocol
template = BinomialABTemplate(ledger)
template.set_protocol(protocol_binding)

# Create Data Stream (2.0% vs 2.1%)
batch_size = max_n_binding // 10
stream = BinomialStream(
    n_per_batch=batch_size,
    arms={"control": 0.020, "treatment": 0.021},
    n_max=max_n_binding + batch_size * 2,
    seed=101,
)

# Run Sequential Monitoring
print(f"Running Simulation 1 (Realistic)...")
for i, batch in enumerate(stream):
    template.update(batch)
    progress = template.report_progress()

    print(
        f"Look {i + 1}: N={progress['sample_n']:,}, Z={progress['z_stat']:.3f}, Status={progress['status']}"
    )

    if progress["status"] != DecisionStatus.CONTINUE_:
        print(f"Stopped early at Look {i + 1}!")
        break

final_res = template.report_result()
print(f"Final Outcome: {final_res['final_status']}")

In [ ]:
# Plot the Trajectory
fig = template.plot_result()
plt.title("Simulation 1: Realistic Lift (2.0% vs 2.1%)")
plt.show()

## 4. Simulation 2: Large Lift (Strong Signal)

Scenario:
*   **Control**: 2.0%
*   **Treatment**: 2.5% (+25% lift)


In [ ]:
# Setup Ledger for Strong simulation
ledger_strong = Ledger(conn, "events_strong")
ledger_strong.ensure()
ledger_strong = ledger_strong.bind(experiment_id="sim_strong_001")

template_strong = BinomialABTemplate(ledger_strong)
template_strong.set_protocol(protocol_binding)  # REUSE Binding protocol

# Create Data Stream (2.0% vs 2.5%)
stream_strong = BinomialStream(
    n_per_batch=batch_size,
    arms={"control": 0.020, "treatment": 0.025},
    n_max=max_n_binding + batch_size * 2,
    seed=999,
)

print(f"Running Simulation 2 (Large Lift)...")
for i, batch in enumerate(stream_strong):
    template_strong.update(batch)
    progress = template_strong.report_progress()

    print(
        f"Look {i + 1}: N={progress['sample_n']:,}, Z={progress['z_stat']:.3f}, Status={progress['status']}"
    )

    if progress["status"] != DecisionStatus.CONTINUE_:
        print(f"Stopped at Look {i + 1}")
        break

final_res_strong = template_strong.report_result()
print(f"Final Outcome: {final_res_strong['final_status']}")

In [ ]:
fig = template_strong.plot_result()
plt.title("Simulation 2: Large Lift (2.0% vs 2.5%)")
plt.show()